# All Models x All Benchmarks: One-Sample Smoke Test

This RunPod-ready notebook runs **one sample for every public model wrapper against every registered benchmark**. Its purpose is integration testing, not model-quality comparison.

Successful pairs are written immediately and skipped on restart. Failed pairs remain visible in the summary and are retried on the next run. The complete matrix includes API models and 32B/72B local checkpoints; a B200-class GPU, a large persistent volume, `OPENAI_API_KEY`, and any required Hugging Face access are recommended.

## 1. Bootstrap RunPod

This locates or clones the repository, configures persistent caches, and installs dependencies while preserving the pod's CUDA-matched PyTorch build.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/samhatescoding/transformers.git"
REPO_BRANCH = None

PERSISTENT_ROOT = Path("/runpod-volume") if Path("/runpod-volume").is_dir() else Path("/workspace")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents, cwd / "transformers", PERSISTENT_ROOT / "transformers"]
REPO_ROOT = next((path.resolve() for path in candidate_roots if (path / "models").is_dir()), None)
if REPO_ROOT is None:
    REPO_ROOT = (PERSISTENT_ROOT / "transformers").resolve()
    command = ["git", "clone"]
    if REPO_BRANCH:
        command += ["--branch", REPO_BRANCH, "--single-branch"]
    command += [REPO_URL, str(REPO_ROOT)]
    subprocess.run(command, check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CACHE_ROOT = PERSISTENT_ROOT / "model-cache"
cache_paths = {
    "HF_HOME": CACHE_ROOT / "huggingface",
    "HF_HUB_CACHE": CACHE_ROOT / "huggingface" / "hub",
    "HF_DATASETS_CACHE": CACHE_ROOT / "huggingface" / "datasets",
    "TRANSFORMERS_CACHE": CACHE_ROOT / "huggingface" / "transformers",
    "TORCH_HOME": CACHE_ROOT / "torch",
}
for variable, path in cache_paths.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(path)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "transformers", "accelerate", "datasets", "openai"],
    check=True,
)

RUN_ROOT = PERSISTENT_ROOT / "all-models-all-benchmarks-smoke-test"
RESULTS_DIR = RUN_ROOT / "results"
FAILED_RESULTS_DIR = RUN_ROOT / "failed-results"
SUMMARY_PATH = RUN_ROOT / "summary.json"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FAILED_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Persistent root: {PERSISTENT_ROOT}")
print(f"Cache: {CACHE_ROOT}")
print(f"Smoke-test output: {RUN_ROOT}")

## 2. Credentials

Set secrets in the RunPod pod environment when possible. Never save real tokens in the notebook.

In [ ]:
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["HF_TOKEN"] = "..."

print("OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "not set")
print("HF_TOKEN:", "set" if os.getenv("HF_TOKEN") else "not set")

## 3. Hardware preflight

The checks warn rather than reject smaller hardware, since Accelerate may offload some models to CPU. Offloading is acceptable for a functional smoke test but can be very slow.

In [ ]:
import torch

free_storage_gib = shutil.disk_usage(PERSISTENT_ROOT).free / 2**30
print(f"Free persistent storage: {free_storage_gib:.1f} GiB")
print(f"CUDA available: {torch.cuda.is_available()}")

total_vram_gib = 0.0
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        vram_gib = props.total_memory / 2**30
        total_vram_gib += vram_gib
        print(f"GPU {index}: {props.name}, {vram_gib:.1f} GiB VRAM")

if not torch.cuda.is_available():
    print("WARNING: local model wrappers require CUDA.")
elif total_vram_gib < 170:
    print("WARNING: less than 170 GiB aggregate VRAM; 72B checkpoints may offload or fail.")
if free_storage_gib < 400:
    print("WARNING: less than 400 GiB free; the complete model cache may exhaust storage.")

## 4. Register the complete matrix

The model registry is derived from `models.__all__`, excluding only the abstract `BaseModel`. This keeps the smoke test synchronized with newly exported wrappers.

In [ ]:
import inspect
import models as model_package
from models import BaseModel
from runners.full_suite import FULL_BENCHMARK_CLASSES

MODEL_CLASSES = {
    export_name: getattr(model_package, export_name)
    for export_name in model_package.__all__
    if export_name != "BaseModel"
    and inspect.isclass(getattr(model_package, export_name, None))
    and issubclass(getattr(model_package, export_name), BaseModel)
}

def make_factory(model_cls):
    return lambda cls=model_cls: cls(max_new_tokens=16)

MODEL_FACTORIES = {
    export_name: make_factory(model_cls)
    for export_name, model_cls in MODEL_CLASSES.items()
}
BENCHMARK_NAMES = [str(cls.benchmark_name) for cls in FULL_BENCHMARK_CLASSES]

print(f"Models: {len(MODEL_FACTORIES)}")
print(f"Benchmarks: {len(FULL_BENCHMARK_CLASSES)}")
print(f"Total pairs: {len(MODEL_FACTORIES) * len(FULL_BENCHMARK_CLASSES)}")
print("Models:", list(MODEL_FACTORIES))
print("Benchmarks:", BENCHMARK_NAMES)

## 5. Validate prerequisites

In [ ]:
from models._openai_vision import OpenAIResponsesVisionModel

api_models = [
    name for name, cls in MODEL_CLASSES.items()
    if issubclass(cls, OpenAIResponsesVisionModel)
]
local_models = [name for name in MODEL_CLASSES if name not in api_models]

if api_models and not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(f"OPENAI_API_KEY is required for API wrappers: {api_models}")
if local_models and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the local model wrappers.")

print(f"API wrappers: {len(api_models)}")
print(f"Local wrappers: {len(local_models)}")
print("Prerequisites passed.")

## 6. Run one sample per pair

`overwrite=False` makes this cell resumable. Result files in this dedicated smoke-test directory represent completed pairs and are skipped on subsequent executions.

In [ ]:
from collections import Counter
from runners.full_suite import run_full_suite

summary = run_full_suite(
    model_factories=MODEL_FACTORIES,
    output_dir=RESULTS_DIR,
    summary_path=SUMMARY_PATH,
    num_samples=1,
    overwrite=False,
    streaming=True,
)

# A runner-level success is insufficient for this smoke test: require exactly
# one prepared sample and no sample-level exception. Quarantine invalid files
# so a restarted notebook retries those pairs instead of skipping them.
import json

quarantined = []
for result_path in RESULTS_DIR.glob("*.json"):
    try:
        payload = json.loads(result_path.read_text(encoding="utf-8"))
        report = payload.get("report", {})
        stats = report.get("stats", {})
        valid = (
            report.get("num_samples") == 1
            and stats.get("success_count") == 1
            and stats.get("failure_count") == 0
        )
        reason = None if valid else (
            f"num_samples={report.get('num_samples')}, "
            f"success_count={stats.get('success_count')}, "
            f"failure_count={stats.get('failure_count')}"
        )
    except Exception as exc:
        valid = False
        reason = f"Invalid result JSON: {type(exc).__name__}: {exc}"

    if not valid:
        failed_path = FAILED_RESULTS_DIR / result_path.name
        result_path.replace(failed_path)
        quarantined.append({"file": result_path.name, "reason": reason})

print(Counter(item["status"] for item in summary))
print(f"Quarantined sample-level failures: {len(quarantined)}")
print(f"Summary: {SUMMARY_PATH}")
print(f"Passing results: {RESULTS_DIR}")
print(f"Failed result payloads: {FAILED_RESULTS_DIR}")

## 7. Coverage report

`PASS` means a one-sample result file exists. `FAIL` means the latest run recorded an error. `MISSING` means the pair has neither a result nor a latest-run status.

In [ ]:
import json
import pandas as pd

latest_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else []
latest_by_pair = {
    (item["model"], item["benchmark"]): item
    for item in latest_summary
}

coverage_rows = []
for model_name in MODEL_FACTORIES:
    for benchmark_name in BENCHMARK_NAMES:
        result_path = RESULTS_DIR / f"{model_name}_{benchmark_name}.json"
        failed_path = FAILED_RESULTS_DIR / f"{model_name}_{benchmark_name}.json"
        latest = latest_by_pair.get((model_name, benchmark_name), {})
        if result_path.exists():
            status = "PASS"
            error = None
        elif failed_path.exists():
            status = "FAIL"
            try:
                failed_payload = json.loads(failed_path.read_text(encoding="utf-8"))
                failed_report = failed_payload.get("report", {})
                failed_stats = failed_report.get("stats", {})
                sample_errors = [
                    item.get("stats", {}).get("error")
                    for item in failed_report.get("results", [])
                    if item.get("stats", {}).get("error")
                ]
                error = "; ".join(sample_errors) or (
                    f"num_samples={failed_report.get('num_samples')}, "
                    f"success_count={failed_stats.get('success_count')}, "
                    f"failure_count={failed_stats.get('failure_count')}"
                )
            except Exception as exc:
                error = f"Invalid failed-result JSON: {type(exc).__name__}: {exc}"
        elif latest.get("status") in {"error", "model_load_error"}:
            status = "FAIL"
            error = latest.get("error")
        else:
            status = "MISSING"
            error = latest.get("error")
        coverage_rows.append({
            "model": model_name,
            "benchmark": benchmark_name,
            "status": status,
            "error": error,
            "result_path": str(result_path) if result_path.exists() else None,
        })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df["status"].value_counts()

In [ ]:
failures_df = coverage_df[coverage_df["status"] != "PASS"].copy()
failures_df

In [ ]:
coverage_pivot = coverage_df.pivot(index="model", columns="benchmark", values="status")
coverage_pivot

## 8. Export the report

In [ ]:
COVERAGE_CSV = RUN_ROOT / "coverage.csv"
FAILURES_CSV = RUN_ROOT / "failures.csv"
coverage_df.to_csv(COVERAGE_CSV, index=False)
failures_df.to_csv(FAILURES_CSV, index=False)
print(f"Coverage report: {COVERAGE_CSV}")
print(f"Failure report: {FAILURES_CSV}")